## Testing the Flip Processor Plugin with ipyleaflet

This notebook demonstrates how to visualize the flip processor results on an interactive map using ipyleaflet.

### Prerequisites

Before running this notebook, ensure that:
1. You have installed the flip processor plugin (`uv pip install -e .` from this directory)
2. You have installed ipyleaflet: `uv pip install ipyleaflet`
3. The vLLM server is running with the plugin enabled
4. The server is accessible at `http://localhost:8000`

In [1]:
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
import requests
import base64
import json
import requests
from ipyleaflet import Map, ImageOverlay, basemaps, basemap_to_tiles
from ipywidgets import Layout
import io
from PIL import Image

### Load and Display Original Image

In [2]:
# Read the original image and get its bounds
input_path = "../samples/India_900498_S2Hand.tif"

with rasterio.open(input_path) as img_src:
    # Read RGB bands
    img_data = img_src.read([3, 2, 1])
    # Transpose to (height, width, channels) for matplotlib
    img_rgb = np.transpose(img_data, (1, 2, 0))
    # Normalize to 0-1 range for display
    img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() - img_rgb.min())
    
    # Get bounds in WGS84 (EPSG:4326) for ipyleaflet
    bounds = img_src.bounds
    src_crs = img_src.crs
    
    # Transform bounds to WGS84 if needed
    if src_crs.to_string() != 'EPSG:4326':
        west, south, east, north = transform_bounds(src_crs, 'EPSG:4326', *bounds)
    else:
        west, south, east, north = bounds.left, bounds.bottom, bounds.right, bounds.top

print(f"Image bounds: {west}, {south}, {east}, {north}")


Image bounds: 93.73524731062194, 26.72236441976023, 93.78124105316887, 26.76835816230715


In [3]:
def array_to_png_url(array, cmap='viridis', alpha=1.0):
    """
    Convert a numpy array to a PNG data URL for ipyleaflet.
    
    Args:
        array: 2D or 3D numpy array
        cmap: matplotlib colormap name (for 2D arrays)
        alpha: transparency (0-1)
    
    Returns:
        PNG data URL string
    """
    print(array.ndim)
    if array.ndim == 2:
        # Single channel - apply colormap
        import matplotlib.pyplot as plt
        colormap = plt.get_cmap(cmap)
        # Normalize to 0-1
        normalized = (array - array.min()) / (array.max() - array.min() + 1e-8)
        # Apply colormap
        colored = colormap(normalized)
        # Convert to uint8
        img_array = (colored[:, :, :3] * 255).astype(np.uint8)
    else:
        # RGB image
        img_array = (array * 255).astype(np.uint8)
    
    # Create PIL Image
    img = Image.fromarray(img_array)
    
    # Convert to PNG bytes
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    buffer.seek(0)
    
    # Encode to base64
    img_base64 = base64.b64encode(buffer.read()).decode()
    
    return f"data:image/png;base64,{img_base64}"

# Calculate center of the image
center_lat = (south + north) / 2
center_lon = (west + east) / 2

# Create the map
m = Map(
    basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik),
    center=(center_lat, center_lon),
    zoom=12,
    layout=Layout(width='100%', height='300px')
)

# Convert original image to PNG URL
img_url = array_to_png_url(img_rgb)

# Add original image as overlay
image_overlay = ImageOverlay(
    url=img_url,
    bounds=((south, west), (north, east)),
    name='Original Image'
)
m.add_layer(image_overlay)

print("Map with original image:")
m

3
Map with original image:


Map(center=[26.74536129103369, 93.7582441818954], controls=(ZoomControl(options=['position', 'zoom_in_text', '…

### Send Inference Request to vLLM

In [ ]:
payload = {
    "data": {
        "data": "https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11/resolve/main/examples/India_900498_S2Hand.tif",
        "indices": [1, 2, 3, 8, 11, 12],
        "data_format": "url",
        "out_data_format": "b64_json",
        "image_format": "tiff",
    },
    "model": "1_run_model_in_vllm",
}

response = requests.post(
    "http://localhost:8000/pooling",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload),
)

if response.status_code == 200:
    result = response.json()
    mask_b64 = result["data"]["data"]
    mask_bytes = base64.b64decode(mask_b64)
    with open("mask.tiff", "wb") as f:
        f.write(mask_bytes)
    print("Segmentation mask saved as mask.tiff")
else:
    print(f"Error: Received status code {response.status_code}")
    print(f"Response: {response.text}")

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /pooling (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 61] Connection refused"))

### Read the Mask

In [4]:
# Read the mask
with rasterio.open('mask.tiff') as mask_src:
    mask = mask_src.read(1)

print(f"Mask shape: {mask.shape}")
print(f"Unique mask values: {np.unique(mask)}")

Mask shape: (512, 512)
Unique mask values: [  0 255]


### Helper Functions for ipyleaflet

In [5]:
def array_to_png_url(array, cmap='viridis', alpha=1.0):
    """
    Convert a numpy array to a PNG data URL for ipyleaflet.
    
    Args:
        array: 2D or 3D numpy array
        cmap: matplotlib colormap name (for 2D arrays)
        alpha: transparency (0-1)
    
    Returns:
        PNG data URL string
    """
    print()
    if array.ndim == 2:
        # Single channel - apply colormap
        import matplotlib.pyplot as plt
        colormap = plt.get_cmap(cmap)
        # Normalize to 0-1
        normalized = (array - array.min()) / (array.max() - array.min() + 1e-8)
        # Apply colormap
        colored = colormap(normalized)
        # Convert to uint8
        img_array = (colored[:, :, :3] * 255).astype(np.uint8)
    else:
        # RGB image
        img_array = (array * 255).astype(np.uint8)
    
    # Create PIL Image
    img = Image.fromarray(img_array)
    
    # Convert to PNG bytes
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    buffer.seek(0)
    
    # Encode to base64
    img_base64 = base64.b64encode(buffer.read()).decode()
    
    return f"data:image/png;base64,{img_base64}"

def create_mask_overlay(mask, alpha=0.6):
    """
    Create a colored mask overlay with transparency.
    
    Args:
        mask: 2D numpy array with mask values
        alpha: transparency for non-zero values
    
    Returns:
        PNG data URL string
    """
    import matplotlib.pyplot as plt
    
    # Use custom color #D02670 for mask overlay
    # Convert hex color to RGB (0-1 range)
    mask_color = np.array([208/255, 38/255, 112/255])  # #D02670
    
    # Create RGBA array
    h, w = mask.shape
    colored = np.zeros((h, w, 4), dtype=np.float32)
    
    # Set RGB channels to the custom color
    colored[:, :, :3] = mask_color
    
    # Set alpha channel - transparent where mask is 0, opaque where mask > 0
    alpha_channel = np.where(mask > 0, alpha, 0.0)
    colored[:, :, 3] = alpha_channel
    
    # Convert to uint8
    img_array = (colored * 255).astype(np.uint8)
    
    # Create PIL Image with alpha
    img = Image.fromarray(img_array, mode='RGBA')
    
    # Convert to PNG bytes
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    buffer.seek(0)
    
    # Encode to base64
    img_base64 = base64.b64encode(buffer.read()).decode()
    
    return f"data:image/png;base64,{img_base64}"

### Create Map with Image and Mask Overlay

In [16]:
# Create flipped image
img_flipped = np.flip(img_rgb, axis=1)

# Create a new map for the flipped version
m2 = Map(
    basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik),
    center=(center_lat, center_lon),
    zoom=12,
    layout=Layout(width='100%', height='300px')
)

# Convert flipped image to PNG URL
flipped_url = array_to_png_url(img_flipped)

# Add flipped image as overlay
flipped_overlay = ImageOverlay(
    url=flipped_url,
    bounds=((south, west), (north, east)),
    name='Flipped Image'
)
m2.add_layer(flipped_overlay)

# Create mask overlay with transparency
mask_url = create_mask_overlay(mask, alpha=0.6)

# Add mask as overlay
mask_overlay = ImageOverlay(
    url=mask_url,
    bounds=((south, west), (north, east)),
    name='Flood Mask'
)
m2.add_layer(mask_overlay)

print("Map with flipped image and mask overlay:")
m2

Map with flipped image and mask overlay:


Map(center=[26.74536129103369, 93.7582441818954], controls=(ZoomControl(options=['position', 'zoom_in_text', '…